# 02 - IEEE 13-node unbalanced feeder

## Objective

Preserve single- and two-phase feeder topology and compare phase-specific voltage magnitudes from direct OpenDSS with the CEPT public CLI.

## Source, assumptions, and units

The source is the IEEE 13-node OpenDSS feeder bundled in the installed CEPT wheel. The comparison point is bus `671`; phase 1, 2, and 3 correspond to A, B, and C. The feeder's declared data are used unchanged. Voltage magnitude is line-to-neutral per unit (`pu`). This is a public demonstrator/reference feeder, not a user's physical project.

## Prediction

The three phase voltages at bus 671 will not be represented safely by one balanced number. The direct OpenDSS and CEPT values should agree within the teaching tolerance when both use the bundled source.

## Action

Solve the bundled source directly, then run `cept study demo unbalanced-load-flow` in a separate exact run directory. The CEPT command streams its output into the originating cell.

## Verification

Read the CEPT solver table from `results.json`, run `cept study verify` on that exact directory, and compare phase identities before comparing values.

## Interpretation

Phase-specific spread is an observation from this solver run. Agreement between two routes through the same public source is a workflow regression check, not independent validation.

## Exercise

Change `BUS_TO_INSPECT` to another named bus present in the direct feeder, inspect its three phase values, and state whether the phase spread increased or decreased. Rerun from a restarted kernel.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT, OpenDSSDirect.py, and the bundled IEEE13 source files. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [ ]:
import hashlib
import importlib.util
import json
import os
import shlex
import subprocess
import sys
import urllib.parse
import urllib.request
from importlib.resources import files
from pathlib import Path

DEFAULT_WHEEL_URL = 'https://github.com/sarutesri/cept-studio-edu/releases/download/v0.2.0-edu.1/cept_power_studio-0.2.0.dev0-py3-none-any.whl'
DEFAULT_WHEEL_SHA256 = 'c7e609a1d9cc85b322bfb615f0c796c7ea5c43815b197289eb555786964478bc'
configured_url = os.environ.get('CEPT_WHEEL_URL')
WHEEL_URL = (DEFAULT_WHEEL_URL if configured_url is None and importlib.util.find_spec('cept') is None else (configured_url or '')).strip()
WHEEL_SHA256 = os.environ.get('CEPT_WHEEL_SHA256', DEFAULT_WHEEL_SHA256 if WHEEL_URL == DEFAULT_WHEEL_URL else '').strip().lower()
if WHEEL_URL:
    if len(WHEEL_SHA256) != 64 or any(character not in '0123456789abcdef' for character in WHEEL_SHA256):
        raise ValueError('CEPT_WHEEL_SHA256 must be the caller-provided 64-character SHA-256')
    wheel_path = Path.cwd() / Path(urllib.parse.urlparse(WHEEL_URL).path).name
    print(f'Downloading caller-provided wheel: {WHEEL_URL}')
    urllib.request.urlretrieve(WHEEL_URL, wheel_path)
    digest = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
    if digest != WHEEL_SHA256:
        raise ValueError(f'wheel hash mismatch: expected {WHEEL_SHA256}, got {digest}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel_path)], check=True)
else:
    print('CEPT_WHEEL_URL not supplied; using the existing installed environment.')
CLI_BIN = Path(sys.executable).parent / ('cept.exe' if os.name == 'nt' else 'cept')
if not CLI_BIN.is_file():
    raise RuntimeError('cept launcher not found next to Python; reinstall the pinned public wheel')
CLI = str(CLI_BIN)
print('CLI: cept --version')
print(subprocess.run([CLI, '--version'], capture_output=True, text=True, check=True).stdout.strip())
def run_cli(*arguments):
    display = 'cept ' + shlex.join([str(argument) for argument in arguments])
    command = [CLI, *[str(argument) for argument in arguments]]
    print('$ ' + display, flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=Path.cwd())
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    returncode = process.wait()
    output = ''.join(lines)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, ['cept', *[str(argument) for argument in arguments]], output=output)
    return json.loads(output) if output.strip().startswith('{') else output

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def show_table(headers, rows):
    print('| ' + ' | '.join(headers) + ' |')
    print('| ' + ' | '.join('---' for _ in headers) + ' |')
    for row in rows:
        print('| ' + ' | '.join(str(value) for value in row) + ' |')

MASTER_DSS = Path(str(files('cept').joinpath('testsystems', 'ieee13', 'IEEE13Nodeckt.dss')))
BUS_TO_INSPECT = '671'


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0


In [ ]:
import opendssdirect as dss

dss.Basic.ClearAll()
dss.Basic.DataPath(str(MASTER_DSS.parent))
dss.Text.Command(f'Redirect \"{MASTER_DSS}\"')
dss.Text.Command('CalcVoltageBases')
dss.Text.Command('Solve')
assert dss.Solution.Converged()
dss.Circuit.SetActiveBus(BUS_TO_INSPECT)
direct_values = dss.Bus.puVmagAngle()
direct_by_phase = {phase: float(direct_values[2 * (phase - 1)]) for phase in (1, 2, 3)}
show_table(['source', 'bus', 'phase', 'voltage magnitude', 'unit'], [('direct OpenDSS', BUS_TO_INSPECT, phase, value, 'pu') for phase, value in direct_by_phase.items()])
assert all(value > 0 for value in direct_by_phase.values())


| source | bus | phase | voltage magnitude | unit |
| --- | --- | --- | --- | --- |
| direct OpenDSS | 671 | 1 | 0.9827953877537872 | pu |
| direct OpenDSS | 671 | 2 | 1.040273943965177 | pu |
| direct OpenDSS | 671 | 3 | 0.9648999396286697 | pu |


In [ ]:
RUN_DIR = Path.cwd() / 'runs' / '02-ieee13-unbalanced'
run_summary = run_cli('study', 'demo', 'unbalanced-load-flow', '--network', 'ieee13', '--out', RUN_DIR, '--force')
verify_summary = run_cli('study', 'verify', RUN_DIR)
results = read_json(RUN_DIR / 'results.json')
cept_rows = [row for row in results['load_flow']['bus_voltages'] if row['bus'].lower() == BUS_TO_INSPECT.lower()]
cept_by_phase = {row['phase']: row['v_pu'] for row in cept_rows}
show_table(['source', 'bus', 'phase', 'voltage magnitude', 'unit'], [('CEPT results.json', BUS_TO_INSPECT, row['phase'], row['v_pu'], 'pu') for row in cept_rows])
show_table(['phase', 'direct OpenDSS pu', 'CEPT pu', 'absolute difference pu'], [(phase, direct_by_phase[phase], cept_by_phase[phase], abs(direct_by_phase[phase] - cept_by_phase[phase])) for phase in (1, 2, 3)])
assert run_summary['status'] == 'PASS'
assert verify_summary['passed'] is True
assert set(cept_by_phase) == {1, 2, 3}
assert max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3)) < 1e-4


$ cept study demo unbalanced-load-flow --network ieee13 --out '<installed-cept>\testsystems\ieee13\runs\02-ieee13-unbalanced' --force


{


  "status": "PASS",


  "claim": "WORKFLOW_VALIDATED",


  "study_type": "unbalanced_load_flow",


...

  "schema": "cept-public-verification-v1",


  "status": "PASS",


  "study_type": "unbalanced_load_flow"


}


| source | bus | phase | voltage magnitude | unit |
| --- | --- | --- | --- | --- |
| CEPT results.json | 671 | 1 | 0.982797 | pu |
| CEPT results.json | 671 | 2 | 1.040275 | pu |
| CEPT results.json | 671 | 3 | 0.964889 | pu |
| phase | direct OpenDSS pu | CEPT pu | absolute difference pu |
| --- | --- | --- | --- |
| 1 | 0.9827953877537872 | 0.982797 | 1.6122462128675963e-06 |
| 2 | 1.040273943965177 | 1.040275 | 1.0560348231436478e-06 |
| 3 | 0.9648999396286697 | 0.964889 | 1.0939628669714985e-05 |


The table is built from the direct solver readback and the CEPT run's persisted `results.json`. Do not average the phases to hide imbalance. The verified public workflow remains a bounded `WORKFLOW_VALIDATED` result and does not establish project or field validation.